# Day 4 · Morning exercises (Exercise A)

**09:45–11:10 · then everyone back in the room at 11:10 for Peer review A**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Keep the folders `data/` and `models/` next to this notebook. The afternoon continues in `day-4-exercises-afternoon.ipynb`.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| A1 | Fit the normal model to one person | 25 min |
| A2 | Joint, marginal and conditional draws | 20 min |
| A3 | Her average versus her next attempt | 20 min |
| — | Exit response and your group's question set | 15 min |

**Roles this block:** coder, checker, two question writers, interviewer, reporter. Rotate them in the afternoon.

**The data.** The Day 1 time game without feedback: each person tried ten times to wait **10 seconds** with eyes closed (we use only the 10-second target today). If the lecturer has placed the class's data as `data/timing-class.csv`, the notebook uses it; otherwise it uses `data/timing-backup.csv`, **synthetic** data for 15 fictional people. The column `elapsed_seconds` is the recorded time.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
DATA <- "data"; MODELS <- "models"     # folders next to this notebook

In [ ]:
# The class's Day 1 data if the lecturer has put them here, otherwise the synthetic backup
file <- if (file.exists(file.path(DATA, "timing-class.csv"))) "timing-class.csv" else "timing-backup.csv"
file
d <- read_csv(file.path(DATA, file), show_col_types = FALSE) |>
  filter(target_seconds == 10, feedback_condition == "no_feedback", !is.na(elapsed_seconds))
d |> count(participant)

## A1 (25 min): fit the normal model to one person

**The model** (*slides "A model with two unknowns" and "The normal model in Stan"*). One person's attempts y₁, …, yₙ:

- yᵢ | μ, σ ~ Normal(μ, σ), independent;
- μ = her long-run average time (seconds), prior μ ~ Normal(10, 2);
- σ = her spread from attempt to attempt (seconds), prior σ ~ Normal⁺(0, 2) (half-normal).

The Stan program is `models/normal.stan`; it also generates `y_new`, a simulated next attempt.

**Tasks.**

1. Choose one person: yourself (your participant ID from Day 1) or the first person in the data. Extract her 10-second times into `y`.
2. Fit the model (4 chains, `seed = 2026`). Check the diagnostics: R-hat, ESS, divergences (*Day 3 slide "Six numbers to check after every fit"*).
3. Report the posterior mean and 90% interval of μ and of σ, with units. In one sentence each: what does μ describe, and what does σ describe?

In [ ]:
me <- d$participant[1]          # or your own ID, for example "G3P07"
y <- d |> filter(participant == me) |> pull(elapsed_seconds)
y

mod_normal <- cmdstan_model(file.path(MODELS, "normal.stan"))
# fit <- mod_normal$sample(data = list(N = length(y), y = y), chains = 4, seed = 2026, refresh = 0)
# fit$summary(c("mu", "sigma"), "mean", ~quantile(.x, c(0.05, 0.95)), "rhat", "ess_bulk", "ess_tail")
# fit$diagnostic_summary()

**Your answer:** write a few sentences here.

## A2 (20 min): joint, marginal and conditional draws

**The idea** (*slides "Joint, marginal and conditional", "Conditional is not marginal" and "With draws, marginalising is free"*). Each row of the draws is a pair (μ⁽ˢ⁾, σ⁽ˢ⁾) from the **joint** posterior. The μ column alone is a sample from the **marginal** posterior of μ. Keeping only the rows where σ is close to one value approximates the **conditional** posterior of μ given that σ.

**Tasks.**

1. Plot the joint draws: μ against σ. Is there a visible relationship between them?
2. Plot a histogram of μ and one of σ (the marginals) and give their 90% intervals.
3. Approximate two conditionals: keep the draws with σ within 0.05 of a small value (the 10% quantile of σ) and of a large value (the 90% quantile). Compare the 90% intervals of μ in these subsets with the marginal interval of μ.
4. Why is the marginal interval of μ the honest one to report?

In [ ]:
# draws <- fit$draws(format = "df")     # one row per draw: mu, sigma, y_new, ...

# Task 1: ggplot(draws, aes(x = mu, y = sigma)) + geom_point(alpha = 0.2)
# Task 2: ggplot(draws, aes(x = mu)) + geom_histogram(bins = 40);  quantile(draws$mu, c(0.05, 0.95))
# Task 3:
# sigma_small <- quantile(draws$sigma, 0.10);  sigma_large <- quantile(draws$sigma, 0.90)
# near_small <- draws |> filter(abs(sigma - sigma_small) < 0.05)
# quantile(near_small$mu, c(0.05, 0.95))

**Your answer:** write a few sentences here.

## A3 (20 min): her average versus her next attempt

**The idea** (*slides "Her average versus her next attempt" and "Questions the draws answer"*). `y_new` was simulated in two steps, inside each draw: take (μ⁽ˢ⁾, σ⁽ˢ⁾), then draw ỹ⁽ˢ⁾ ~ Normal(μ⁽ˢ⁾, σ⁽ˢ⁾).

**Tasks.**

1. Give the 90% interval of μ and of `y_new`. Plot both histograms in one figure.
2. Compute P(μ < 10 | y) and P(ỹ < 10 | y). Why do they differ?
3. Compute the probability that her next attempt is within half a second of 10 s.
4. Which of these numbers would change most if she made 100 attempts instead of 10, and which hardly at all? (*slide "Worked example: answers", morning*)

In [ ]:
# Task 1
# quantile(draws$mu, c(0.05, 0.95));  quantile(draws$y_new, c(0.05, 0.95))
# ggplot(draws) +
#   geom_histogram(aes(x = y_new, y = after_stat(density), fill = "her next attempt (y_new)"), bins = 60, alpha = 0.6) +
#   geom_histogram(aes(x = mu,    y = after_stat(density), fill = "her average (mu)"),         bins = 60, alpha = 0.6) +
#   geom_vline(xintercept = 10, linetype = "dashed") +
#   scale_fill_manual(values = c("her next attempt (y_new)" = "salmon", "her average (mu)" = "navy"), name = NULL) +
#   theme(legend.position = "top") +
#   labs(x = "seconds", y = "density")

# Tasks 2 and 3: mean(draws$mu < 10);  mean(draws$y_new < 10);  mean(abs(draws$y_new - 10) < 0.5)

**Your answer:** write a few sentences here.

*Optional:* fit the same model to a second person and compare the two. Because the two fits are independent, P(μ₁ < μ₂) can be estimated as `mean(draws1$mu < draws2$mu)`. Is one of them clearly faster?

## Exit response

Write 80–120 words: What is a marginal posterior, and how do you get it from draws? Why is the interval for her next attempt wider than the interval for her average?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 11:10 for Peer review A.** Save this notebook (Ctrl/Cmd + S). After lunch, open `day-4-exercises-afternoon.ipynb`.